# CogMem Phase 2 — Q-Weighted DoRA Training (BigCodeBench-Hard)

Train **DoRA** on Qwen2.5:3b using BigCodeBench-Hard (148 tasks) episodes.

**Why 8-bit + DoRA:** DoRA + 4-bit is broken in PEFT. 8-bit uses `Linear8bitLt` which DoRA supports.

**Requires:** `memory_bank_hard_*.json` from eval notebook (paperspace_bigcode.ipynb)

**Flow:**
- Cells 1-4: Train (restart kernel after Cell 1)
- Cell 4b: Emergency save if you want to stop early
- Cell 5: Merge adapter into full model
- Cell 6: **Restart kernel first!** Then start Ollama + load models
- Cell 7: Evaluate on BigCodeBench-Hard (148 tasks)
- Cell 8: Package results

In [10]:
# Cell 1: Install deps (DO NOT touch torch)
!pip install "transformers==4.43.4" "peft==0.13.2" "accelerate==0.33.0" "bitsandbytes==0.43.3" "datasets==2.20.0" "huggingface-hub>=0.24" "pydantic>=2.0" pyyaml -q
!python3 -c "import torch; print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
!python3 -c "from transformers import Trainer; print('Trainer OK')"
print("Restart kernel, then run Cell 2")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


torch 2.1.1+cu121, CUDA: True


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2026-04-04 18:29:03.402495: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-04 18:29:03.439025: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-04 18:29:03.439087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-04 18:29:03.440233: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-04 18:29:03.446059: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
# Cell 2: Verify imports + HuggingFace login
import torch, transformers, peft
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

from huggingface_hub import login
import os
token = os.environ.get("HF_TOKEN", "")
if token:
    login(token=token)
else:
    print("Set HF_TOKEN env var or paste token below:")
    login()

In [ ]:
# Cell 3: Build training data from Hard-148 episodes
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json, glob
from pathlib import Path

# Find all Hard memory banks (one per model that was evaluated)
mb_files = sorted(glob.glob("/notebooks/CogMem/results/memory_bank_hard_*.json"))
if not mb_files:
    # Try checkpoint files directly
    ckpt_files = sorted(glob.glob("/notebooks/bigcode_hard_*.jsonl"))
    if ckpt_files:
        print("Converting checkpoints to memory banks...")
        for ckpt in ckpt_files:
            episodes = []
            with open(ckpt) as f:
                for line in f:
                    if line.strip():
                        episodes.append(json.loads(line))
            model_name = Path(ckpt).stem.replace("bigcode_hard_", "")
            mb_path = f"/notebooks/CogMem/results/memory_bank_hard_{model_name}.json"
            Path(mb_path).parent.mkdir(parents=True, exist_ok=True)
            with open(mb_path, "w") as f:
                json.dump(episodes, f, indent=2)
            mb_files.append(mb_path)
            print(f"  {mb_path}: {len(episodes)} episodes")
    else:
        raise FileNotFoundError("Run eval notebook first! Need memory_bank_hard_*.json")

# Combine all episodes from all models
all_episodes = []
for mb_path in mb_files:
    with open(mb_path) as f:
        eps = json.load(f)
    passed = sum(1 for ep in eps if ep["success"])
    model = eps[0].get("model", Path(mb_path).stem) if eps else "?"
    print(f"  {model}: {passed}/{len(eps)} ({passed/len(eps):.1%})")
    all_episodes.extend(eps)

print(f"\nTotal episodes: {len(all_episodes)}")

# Build Q-weighted training data
from scripts.build_training_data_bigcode import build_training_data

# Save combined memory bank
MB_COMBINED = "/notebooks/CogMem/results/memory_bank_bigcode_hard.json"
with open(MB_COMBINED, "w") as f:
    json.dump(all_episodes, f, indent=2)

build_training_data(MB_COMBINED, "/notebooks/CogMem/results/training_bigcode_hard.jsonl")

In [ ]:
# Cell 4: Train DoRA (8-bit) on Qwen2.5:3b — Hard-148 episodes, Q-weighted
import shutil, os
ADAPTER_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_dora"
for d in [ADAPTER_DIR, "/notebooks/CogMem/adapters/qwen_bigcode_lora"]:
    if os.path.exists(d) and not os.path.exists(os.path.join(d, "adapter_config.json")):
        shutil.rmtree(d)
        print(f"Cleaned stale: {d}")

import json, torch, gc
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)

os.environ["TRANSFORMERS_NO_FLASH_ATTENTION"] = "1"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
JSONL_PATH = "/notebooks/CogMem/results/training_bigcode_hard.jsonl"

with open(JSONL_PATH) as f:
    raw_data = [json.loads(line) for line in f]
print(f"Training samples: {len(raw_data)} (Hard-148 episodes)")

# 8-bit quantization — DoRA works with Linear8bitLt (not Linear4bit)
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map={"": 0},
    torch_dtype=torch.float16, attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)

dora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    use_dora=True,
)

gc.collect()
torch.cuda.empty_cache()

model = get_peft_model(model, dora_config)
model.print_trainable_parameters()

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    tok = tokenizer(text, truncation=True, max_length=512, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

dataset = Dataset.from_list(raw_data).map(format_chat, remove_columns=["messages"])
print(f"Tokenized: {len(dataset)} examples")

gc.collect()
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ADAPTER_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        fp16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        seed=42,
        gradient_checkpointing=True,
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

has_checkpoints = os.path.exists(ADAPTER_DIR) and any(d.startswith("checkpoint") for d in os.listdir(ADAPTER_DIR))
print(f"Starting DoRA training (resume={has_checkpoints})...")
trainer.train(resume_from_checkpoint=has_checkpoints)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"DoRA adapter saved to {ADAPTER_DIR}")

In [ ]:
# Cell 4b: Emergency save — run this to force-save the current model NOW
# (Run in a separate cell while Cell 4 is still training, or after you interrupt it)

ADAPTER_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_dora"

# Save adapter weights
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Model saved to {ADAPTER_DIR}")
print("You can now interrupt Cell 4 and run Cell 5 to merge.")

In [ ]:
# Cell 5: Merge DoRA adapter + create Ollama model for evaluation
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_dora"
MERGED_DIR = "/notebooks/cogmem_qwen_bigcode_merged"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Loading DoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_DIR)

print("Merging (magnitude+direction fold back — zero inference overhead)...")
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged to {MERGED_DIR}")

# Install Ollama if not already running
!which ollama || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh)

import subprocess, time, os
try:
    subprocess.check_output(["ollama", "list"], timeout=5)
    print("Ollama already running")
except Exception:
    proc = subprocess.Popen(
        ["ollama", "serve"],
        env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
        stdout=open("/tmp/ollama.log", "w"),
        stderr=subprocess.STDOUT,
    )
    time.sleep(5)

# IMPORTANT: Must include Qwen chat template — without it Ollama generates garbage
with open("/notebooks/Modelfile.cogmem_bigcode", "w") as f:
    f.write(f"""FROM {MERGED_DIR}
TEMPLATE \"\"\"{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"
PARAMETER stop <|im_end|>
PARAMETER temperature 0
""")
!ollama create cogmem-qwen-bigcode -f /notebooks/Modelfile.cogmem_bigcode
!ollama list
print("cogmem-qwen-bigcode model ready for evaluation!")

In [ ]:
# Cell 6: Start Ollama + load models for evaluation
# IMPORTANT: After training, restart kernel first to free GPU VRAM!
# Then run this cell before Cell 7 (evaluation).

import subprocess, time, os

# Kill any stale ollama process
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

# Start Ollama server
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

# Verify server is up
!curl -sf http://localhost:11434/api/tags > /dev/null && echo "Ollama server OK" || echo "ERROR: Ollama not responding"

# Pull base model
!ollama pull qwen2.5:3b

# Recreate cogmem model with proper chat template
MERGED_DIR = "/notebooks/cogmem_qwen_bigcode_merged"
with open("/notebooks/Modelfile.cogmem_bigcode", "w") as f:
    f.write(f"""FROM {MERGED_DIR}
TEMPLATE \"\"\"{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"
PARAMETER stop <|im_end|>
PARAMETER temperature 0
""")

!ollama create cogmem-qwen-bigcode -f /notebooks/Modelfile.cogmem_bigcode
!ollama list

# Quick smoke test
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
resp = client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Write a Python function that adds two numbers."}],
    max_tokens=200, temperature=0,
)
print(f"\nSmoke test response:\n{resp.choices[0].message.content[:300]}")
print("\nOllama ready! Now run Cell 7 (evaluation).")

In [ ]:
# Cell 7: Evaluate on BigCodeBench-Hard (148 tasks)
import os
for f in ["/notebooks/eval_hard_qwen2_5_3b.jsonl", "/notebooks/eval_hard_cogmem_qwen_bigcode.jsonl"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted stale checkpoint: {f}")

import json, sys
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Load Hard tasks
TASKS_PATH = "/notebooks/bigcodebench_hard_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench-hard", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + "\n")

eval_tasks = []
with open(TASKS_PATH) as f:
    for line in f:
        eval_tasks.append(json.loads(line.strip()))

print(f"Evaluating on {len(eval_tasks)} Hard tasks")

results = {}
for model_name in ["qwen2.5:3b", "cogmem-qwen-bigcode"]:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    passed = 0
    total = 0
    checkpoint = f"/notebooks/eval_hard_{model_name.replace(':', '_').replace('-', '_')}.jsonl"
    
    done_ids = set()
    if Path(checkpoint).exists():
        with open(checkpoint) as f:
            for line in f:
                if line.strip():
                    ep = json.loads(line)
                    done_ids.add(ep["task_id"])
                    total += 1
                    if ep["passed"]: passed += 1
        print(f"Resumed: {len(done_ids)} done")
    
    remaining = [t for t in eval_tasks if t["task_id"] not in done_ids]
    
    for i, task in enumerate(remaining):
        try:
            messages = format_messages(task, use_instruct=True)
            resp = client.chat.completions.create(
                model=model_name, messages=messages,
                max_tokens=2048, temperature=0,
            )
            response = resp.choices[0].message.content
            code = extract_code(response)
            result = evaluate_solution(task, code, timeout=30, mode="subprocess")
            task_passed = result["passed"]
            if total < 3:
                status = "PASS" if task_passed else "FAIL"
                print(f"  {task['task_id']}: {status}")
        except Exception as e:
            task_passed = False
        
        total += 1
        if task_passed: passed += 1
        
        with open(checkpoint, "a") as f:
            f.write(json.dumps({"task_id": task["task_id"], "passed": task_passed}) + "\n")
        
        if (i + 1) % 20 == 0:
            print(f"  [{total}/{len(eval_tasks)}] Pass: {passed}/{total} ({passed/total:.1%})")
    
    results[model_name] = {"passed": passed, "total": total, "rate": passed / total if total > 0 else 0}

# Print comparison
print(f"\n{'='*60}")
print(f"{'BigCodeBench-Hard RESULTS':^60}")
print(f"{'='*60}")
print(f"{'Model':<30} {'Passed':>8} {'Total':>8} {'Rate':>10}")
print(f"{'-'*60}")
for model_name, r in results.items():
    print(f"{model_name:<30} {r['passed']:>8} {r['total']:>8} {r['rate']:>9.1%}")

if len(results) == 2:
    rates = list(results.values())
    improvement = rates[1]["rate"] - rates[0]["rate"]
    print(f"\n{'Improvement':<30} {'':>8} {'':>8} {improvement:>+9.1%}")

with open("/notebooks/bigcode_hard_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to /notebooks/bigcode_hard_eval_results.json")

In [ ]:
# Cell 8: Package adapter for download / next cycle
!tar czf /notebooks/cogmem_bigcode_dora_adapter.tar.gz \
    -C /notebooks/CogMem adapters/qwen_bigcode_dora/
!ls -lh /notebooks/cogmem_bigcode_dora_adapter.tar.gz
!ls -lh /notebooks/bigcode_eval_results.json
print("Download:")
print("  cogmem_bigcode_dora_adapter.tar.gz — DoRA adapter for Cycle 2+")
print("  bigcode_eval_results.json — evaluation results")